# Smart City RAG Assistant
### Agentic AI Engineering Bootcamp — RAG Foundations

This notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** system for the Smart Cities domain.

The project demonstrates foundational concepts used in Agentic AI systems:

- External knowledge ingestion
- Document chunking
- Semantic embeddings
- Vector search with FAISS
- Retrieval of relevant context
- Grounded LLM generation
- Retrieval inspection and validation

> **Lab requirements:** 6 web documents, 3 queries, and at least 6 retrieved chunks per query.

This notebook focuses on **RAG as an Agentic AI foundation**. It does not claim to be a fully autonomous agent; a later extension can expose this retriever as a tool inside a LangGraph or tool-calling agent.


## System Architecture

```text
6 Smart City Webpages
        ↓
  WebBaseLoader
        ↓
  6 Documents
        ↓
RecursiveCharacterTextSplitter
        ↓
   Text Chunks
        ↓
 OpenAI Embeddings
        ↓
   FAISS Vector Store
        ↓
 Retriever (Top k = 6)
        ↓
      User Query
        ↓
Top 6 Relevant Chunks
        ↓
 Context + Question
        ↓
   GPT-4.1-mini
        ↓
  Grounded Answer
```


## 1. Install Dependencies


In [1]:
%pip install -q langchain langchain-community langchain-openai langchain-classic faiss-cpu tiktoken beautifulsoup4 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Configure the OpenAI API Key

In [3]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")  # works only in the Colab browser UI
    except Exception:
        api_key = None
    if not api_key:
        api_key = getpass("Paste your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = api_key.strip()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not set.")

# Helps identify requests made by WebBaseLoader.
os.environ.setdefault("USER_AGENT", "smart-city-rag-bootcamp/1.0")

print("OpenAI API key is configured.")


OpenAI API key is configured.


## 3. Define the Smart City Knowledge Sources

The knowledge base uses six public webpages covering complementary Smart City perspectives:

1. **IBM** — smart-city concepts and technologies
2. **Riyadh Municipality** — real Smart City initiatives in Riyadh
3. **ITU** — smart and sustainable cities
4. **World Bank** — smart technology and sustainable development
5. **NIST** — IoT / cyber-physical systems for Smart Cities
6. **OECD** — Smart Cities and inclusive growth

Each webpage represents one source document before chunking.


In [4]:
SOURCES = [
    {
        "name": "IBM — What is a Smart City?",
        "url": "https://www.ibm.com/think/topics/smart-city",
        "focus": "Smart-city concepts, AI, IoT, transport, energy, and infrastructure",
    },
    {
        "name": "Riyadh Municipality — Riyadh Smart City",
        "url": "https://alriyadh.gov.sa/en/content/smart-cities-initiatives",
        "focus": "Riyadh smart-city initiatives and urban services",
    },
    {
        "name": "ITU — Smart Sustainable Cities",
        "url": "https://www.itu.int/web/pp-18/en/backgrounder/smart-sustainable-cities",
        "focus": "ICT, sustainability, infrastructure, and smart-city development",
    },
    {
        "name": "World Bank — Smart Technology for Sustainable Development",
        "url": "https://www.worldbank.org/en/news/feature/2021/01/18/harnessing-smart-technology-for-sustainable-development-in-developing-countries-tdlc",
        "focus": "Smart technology and sustainable urban development",
    },
    {
        "name": "NIST — CPS / IoT for Smart Cities",
        "url": "https://www.nist.gov/programs-projects/cyber-physical-systemsinternet-things-smart-cities",
        "focus": "IoT, cyber-physical systems, standards, and city services",
    },
    {
        "name": "OECD — Smart Cities and Inclusive Growth",
        "url": "https://www.oecd.org/en/about/programmes/the-oecd-programme-on-smart-cities-and-inclusive-growth0.html",
        "focus": "Well-being, inclusion, public services, sustainability, and digitalisation",
    },
]

print("Configured sources:", len(SOURCES))
for i, source in enumerate(SOURCES, 1):
    print(f"{i}. {source['name']}")


Configured sources: 6
1. IBM — What is a Smart City?
2. Riyadh Municipality — Riyadh Smart City
3. ITU — Smart Sustainable Cities
4. World Bank — Smart Technology for Sustainable Development
5. NIST — CPS / IoT for Smart Cities
6. OECD — Smart Cities and Inclusive Growth


## 4. Load the Six Web Documents

`WebBaseLoader` retrieves the text from each webpage. The code below deliberately loads each URL separately so that the notebook can verify that all six sources were successfully ingested.


In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

documents = []
load_failures = []

for source in SOURCES:
    try:
        loaded = WebBaseLoader(source["url"]).load()

        if not loaded:
            raise ValueError("No document content was returned.")

        # One source URL should represent one source document.
        # If a loader ever returns multiple pieces, combine them into one document.
        combined_text = "\n\n".join(doc.page_content for doc in loaded)

        metadata = dict(loaded[0].metadata)
        metadata.update(
            {
                "source": source["url"],
                "source_name": source["name"],
                "focus": source["focus"],
            }
        )

        documents.append(
            Document(
                page_content=combined_text,
                metadata=metadata,
            )
        )

    except Exception as exc:
        load_failures.append((source["name"], str(exc)))

print(f"Successfully loaded: {len(documents)} documents")

if load_failures:
    print("\nLoad failures:")
    for name, error in load_failures:
        print(f"- {name}: {error}")


/tmp/ipykernel_2557/141494629.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


Successfully loaded: 6 documents


### Verify the 6-document requirement

In [8]:
assert len(documents) == 6, (
    f"Expected 6 documents, but loaded {len(documents)}. "
    "Review any load failures printed above."
)

print("Requirement satisfied: 6 documents loaded.")


Requirement satisfied: 6 documents loaded.


### Inspect the loaded sources

In [9]:
for i, doc in enumerate(documents, 1):
    print(f"Document {i}")
    print("Name:  ", doc.metadata.get("source_name"))
    print("Source:", doc.metadata.get("source"))
    print("Chars: ", len(doc.page_content))
    print("-" * 90)


Document 1
Name:   IBM — What is a Smart City?
Source: https://www.ibm.com/think/topics/smart-city
Chars:  16229
------------------------------------------------------------------------------------------
Document 2
Name:   Riyadh Municipality — Riyadh Smart City
Source: https://alriyadh.gov.sa/en/content/smart-cities-initiatives
Chars:  10392
------------------------------------------------------------------------------------------
Document 3
Name:   ITU — Smart Sustainable Cities
Source: https://www.itu.int/web/pp-18/en/backgrounder/smart-sustainable-cities
Chars:  13553
------------------------------------------------------------------------------------------
Document 4
Name:   World Bank — Smart Technology for Sustainable Development
Source: https://www.worldbank.org/en/news/feature/2021/01/18/harnessing-smart-technology-for-sustainable-development-in-developing-countries-tdlc
Chars:  8497
------------------------------------------------------------------------------------------
Doc

## 5. Chunk the Documents

Large webpages are split into smaller overlapping chunks before embedding.

- `chunk_size = 500`
- `chunk_overlap = 50`

The overlap helps preserve context across chunk boundaries.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Generated chunks:  ", len(chunks))

assert len(chunks) >= 6, "At least 6 chunks are required for top-6 retrieval."


Original documents: 6
Generated chunks:   167


### Inspect example chunks

In [11]:
for i, chunk in enumerate(chunks[:3], 1):
    print(f"Chunk {i}")
    print("Source:", chunk.metadata.get("source_name"))
    print(chunk.page_content[:500].strip())
    print("\n" + "-" * 90 + "\n")


Chunk 1
Source: IBM — What is a Smart City?
What is a Smart City? | IBM
















































































































Home





Think





Topics




Smart City








                        
                        


  
  
    
    What is a smart city?

  





    


                    





























        


  
  
    
    Authors

  





    


    















Alice Gomstyn

Staff Writer
IBM Think















Alexandra Jonker

------------------------------------------------------------------------------------------

Chunk 2
Source: IBM — What is a Smart City?
Alexandra Jonker

Staff Editor
IBM Think










        What is a smart city?
    


A smart city is an urban area where technology and data collection help improve quality of life as well as the sustainability and efficiency of city operations. Smart city technologies used by local governments include information and communicat

## 6. Create Embeddings

Each chunk is converted into a numerical vector using `text-embedding-3-small`.

A local embedding cache is used so repeated runs do not need to recompute every unchanged embedding.


In [13]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

CACHE_PATH = "./cache/"
os.makedirs(CACHE_PATH, exist_ok=True)

store = LocalFileStore(CACHE_PATH)

core_embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

embedder = CacheBackedEmbeddings.from_bytes_store(
    core_embeddings_model,
    store,
    namespace=core_embeddings_model.model,
)

print("Embedding model and local cache are ready.")


Embedding model and local cache are ready.


## 7. Build the FAISS Vector Store

FAISS stores the vector representation of every chunk and enables fast semantic similarity search.


In [14]:
from langchain_community.vectorstores import FAISS
import time

start_time = time.time()

vectorstore = FAISS.from_documents(
    chunks,
    embedder,
)

elapsed = time.time() - start_time

print(f"FAISS vector store created in {elapsed:.2f} seconds.")
print("Vectors stored:", vectorstore.index.ntotal)


FAISS vector store created in 5.65 seconds.
Vectors stored: 167


## 8. Configure Top-6 Retrieval

`k=6` means that, for every user query, FAISS returns the **six most semantically relevant chunks** from the entire vector store.

The six chunks do **not** need to come from six different webpages. Retrieval is based on relevance.


In [16]:
TOP_K = 6

retriever = vectorstore.as_retriever(
    search_kwargs={"k": TOP_K}
)

print(f"Retriever configured with k={TOP_K}.")


Retriever configured with k=6.


## 9. Define the Three Evaluation Queries

The queries test different parts of the knowledge base:

1. General Smart City technology
2. A Saudi / Riyadh-specific use case
3. Sustainability, inclusion, benefits, and challenges


In [17]:
queries = [
    "What technologies are commonly used in smart cities and how do they improve urban services?",
    "How is Riyadh using artificial intelligence, IoT, and smart technologies to improve city services?",
    "What are the main benefits and challenges of developing inclusive and sustainable smart cities?",
]

assert len(queries) == 3, f"Expected 3 queries, found {len(queries)}."

print("Requirement satisfied: 3 queries defined.")
for i, query in enumerate(queries, 1):
    print(f"{i}. {query}")


Requirement satisfied: 3 queries defined.
1. What technologies are commonly used in smart cities and how do they improve urban services?
2. How is Riyadh using artificial intelligence, IoT, and smart technologies to improve city services?
3. What are the main benefits and challenges of developing inclusive and sustainable smart cities?


## 10. Inspect Retrieval Before Calling the LLM

This step is important because it shows what the vector search actually selected.

For each query, the notebook records:

- Number of retrieved chunks
- Rank of each retrieved chunk
- Source webpage
- Preview of the retrieved text

This makes the retrieval process observable rather than treating RAG as a black box.


In [18]:
import pandas as pd

retrieval_rows = []
retrieved_by_query = {}

for query_id, query in enumerate(queries, 1):
    retrieved_docs = retriever.invoke(query)
    retrieved_by_query[query_id] = retrieved_docs

    assert len(retrieved_docs) >= 6, (
        f"Query {query_id} returned only {len(retrieved_docs)} chunks."
    )

    for rank, doc in enumerate(retrieved_docs, 1):
        retrieval_rows.append(
            {
                "query_id": query_id,
                "query": query,
                "rank": rank,
                "source_name": doc.metadata.get("source_name"),
                "source_url": doc.metadata.get("source"),
                "chunk_preview": doc.page_content[:220].replace("\n", " "),
            }
        )

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df


,query_id,query,rank,source_name,source_url,chunk_preview
0,1,What technologies are commonly used in smart c...,1,IBM — What is a Smart City?,https://www.ibm.com/think/topics/smart-city,Technologies to collect data—including real-ti...
1,1,What technologies are commonly used in smart c...,2,IBM — What is a Smart City?,https://www.ibm.com/think/topics/smart-city,Smart city technologies New technologie...
2,1,What technologies are commonly used in smart c...,3,ITU — Smart Sustainable Cities,https://www.itu.int/web/pp-18/en/backgrounder/...,Although cities where all urban systems and se...
3,1,What technologies are commonly used in smart c...,4,IBM — What is a Smart City?,https://www.ibm.com/think/topics/smart-city,Smart city technologies used in smart transpor...
4,1,What technologies are commonly used in smart c...,5,IBM — What is a Smart City?,https://www.ibm.com/think/topics/smart-city,"Areas of city operations where ICT, IoT and ot..."
5,1,What technologies are commonly used in smart c...,6,ITU — Smart Sustainable Cities,https://www.itu.int/web/pp-18/en/backgrounder/...,Challenges and solutions Smart sustainable cit...
6,2,"How is Riyadh using artificial intelligence, I...",1,Riyadh Municipality — Riyadh Smart City,https://alriyadh.gov.sa/en/content/smart-citie...,"through sensors to determine the used, the non..."
7,2,"How is Riyadh using artificial intelligence, I...",2,Riyadh Municipality — Riyadh Smart City,https://alriyadh.gov.sa/en/content/smart-citie...,of Riyadh into a fully integrated smart city. ...
8,2,"How is Riyadh using artificial intelligence, I...",3,Riyadh Municipality — Riyadh Smart City,https://alriyadh.gov.sa/en/content/smart-citie...,Drones This initiative utilizes drones to capt...
9,2,"How is Riyadh using artificial intelligence, I...",4,Riyadh Municipality — Riyadh Smart City,https://alriyadh.gov.sa/en/content/smart-citie...,The municipality is working on expanding the s...


### Retrieval-count validation

In [19]:
retrieval_summary = (
    retrieval_df
    .groupby("query_id")
    .agg(
        retrieved_chunks=("rank", "count"),
        unique_sources=("source_url", "nunique"),
    )
)

retrieval_summary


,retrieved_chunks,unique_sources
query_id,,
1,6,2
2,6,1
3,6,3


Expected result: each query should show **6 retrieved chunks**.

`unique_sources` may be less than 6 because multiple highly relevant chunks can come from the same webpage.


## 11. Configure the LLM

The generation model receives the user question together with the context selected by the retriever.


In [21]:
from langchain_openai import ChatOpenAI
from langchain_classic.callbacks import StdOutCallbackHandler

handler = StdOutCallbackHandler()

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0,
    streaming=False,
    callbacks=[handler],
)

print("LLM is ready.")


LLM is ready.


## 12. Build the RetrievalQA Chain

`return_source_documents=True` keeps the retrieved evidence available with every answer, which helps inspect and trace the RAG output.


In [22]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    callbacks=[handler],
    return_source_documents=True,
)

print("RetrievalQA chain is ready.")


RetrievalQA chain is ready.


## 13. Run the Three RAG Queries

In [23]:
rag_results = []

for query_id, query in enumerate(queries, 1):
    print("\n" + "=" * 100)
    print(f"QUERY {query_id}")
    print("=" * 100)
    print(query)

    response = qa_chain.invoke({"query": query})

    answer = response["result"]
    source_documents = response["source_documents"]

    print("\nANSWER")
    print("-" * 100)
    print(answer)

    print("\nRETRIEVAL")
    print("-" * 100)
    print("Retrieved chunks:", len(source_documents))

    for rank, doc in enumerate(source_documents, 1):
        print(f"\nChunk {rank}: {doc.metadata.get('source_name')}")
        print("Source:", doc.metadata.get("source"))
        print(doc.page_content[:250].replace("\n", " "))

    rag_results.append(
        {
            "query_id": query_id,
            "query": query,
            "answer": answer,
            "retrieved_chunks": len(source_documents),
            "unique_sources": len(
                {doc.metadata.get("source") for doc in source_documents}
            ),
        }
    )



QUERY 1
What technologies are commonly used in smart cities and how do they improve urban services?


> Entering new RetrievalQA chain...

> Finished chain.

ANSWER
----------------------------------------------------------------------------------------------------
Smart cities commonly use a range of technologies, particularly information and communication technology (ICT), the Internet of Things (IoT), artificial intelligence (AI), and smart grids and meters. These technologies improve urban services in several ways:

1. **Data Collection and Real-Time Insights:** Technologies collect real-time data from various urban systems, enabling local governments to gain data-driven insights. This helps improve urban planning and the deployment of city services.

2. **Transportation:** Smart transportation systems use AI and real-time traffic data to optimize traffic flow, manage signal timing at intersections, and predict vehicle breakdowns for proactive repairs. Smart parking systems assess

## 14. Final Lab Validation

This cell makes the assignment requirements explicit and reproducible.


In [ ]:
results_df = pd.DataFrame(rag_results)

documents_pass = len(documents) == 6
queries_pass = len(queries) == 3
retrieval_pass = all(
    result["retrieved_chunks"] >= 6
    for result in rag_results
)

validation_df = pd.DataFrame(
    [
        {
            "requirement": "6 web documents",
            "observed": len(documents),
            "status": "PASS" if documents_pass else "FAIL",
        },
        {
            "requirement": "3 queries",
            "observed": len(queries),
            "status": "PASS" if queries_pass else "FAIL",
        },
        {
            "requirement": "At least 6 retrieved chunks per query",
            "observed": ", ".join(
                str(result["retrieved_chunks"])
                for result in rag_results
            ),
            "status": "PASS" if retrieval_pass else "FAIL",
        },
    ]
)

validation_df


In [ ]:
assert documents_pass, "6-document requirement failed."
assert queries_pass, "3-query requirement failed."
assert retrieval_pass, "At least one query retrieved fewer than 6 chunks."

print("✓ All lab requirements passed.")


## 15. What This Project Demonstrates

This notebook demonstrates a complete RAG lifecycle:

```text
Web Data
   ↓
Document Loading
   ↓
Chunking
   ↓
Embedding
   ↓
Vector Storage
   ↓
Semantic Retrieval
   ↓
Context Augmentation
   ↓
LLM Generation
   ↓
Source Inspection
```



### Skills demonstrated

- LangChain document loading
- Text chunking
- OpenAI embeddings
- FAISS vector search
- Semantic retrieval
- Retrieval-Augmented Generation
- Source tracing
- Basic retrieval validation
- Grounding LLM responses with external knowledge
